In [1]:
import datetime
import pandas as pd
import plotly.graph_objects as go
import pytz

# =====================================================================
# 1. PROGRAMMATIC DATASET CREATION
# =====================================================================
# Comprehensive data covering required metrics to satisfy filtering criteria
raw_data = {
    "App": [
        "Subway Surfers", "Instagram", "Google Drive", "WhatsApp", "Dropbox",
        "Candy Crush Saga", "Pinterest", "Slack", "Microsoft Word", "Flipboard",
        "Skype", "My Fitness Pal", "Duolingo", "PicsArt Photo Editor",
        "Launcher iOS", "VLC for Android", "Files by Google", "Evernote"
    ],
    "Category": [
        "GAME", "SOCIAL", "PRODUCTIVITY", "COMMUNICATION", "BUSINESS",
        "GAME", "SOCIAL", "BUSINESS", "PRODUCTIVITY", "NEWS_AND_MAGAZINES",
        "COMMUNICATION", "HEALTH_AND_FITNESS", "EDUCATION", "PHOTOGRAPHY",
        "ART_AND_DESIGN", "VIDEO_PLAYERS", "PRODUCTIVITY", "PRODUCTIVITY"
    ],
    "Rating": [4.5, 4.3, 4.4, 4.4, 3.9, 4.6, 4.2, 3.8, 4.1, 3.5, 4.0, 4.5, 4.7, 4.3, 4.8, 4.3, 4.6, 4.4],
    "Reviews": [
        27722264, 66577313, 4144655, 69119316, 1860844, 
        22419455, 4305441, 150000, 3450000, 1200000, 
        11000000, 120000, 6500000, 8500000, 150000, 
        1400000, 5000000, 1600000
    ],
    "Installs": [
        1000000000, 1000000000, 1000000000, 1000000000, 500000000,
        500000000, 500000000, 10000000, 1000000000, 500000000,
        1000000000, 50000000, 100000000, 500000000,
        10000000, 100000000, 500000000, 100000000
    ],
    "Size": [
        "76M", "Varies with device", "12M", "Varies with device", "61M",
        "74M", "Varies with device", "22M", "50M", "15M",
        "32M", "45M", "28M", "38M", "14M", "25M", "11M", "42M"
    ],
    "Last Updated": [
        "January 15, 2026", "January 20, 2026", "January 12, 2026", "March 18, 2026", "January 05, 2026",
        "January 22, 2026", "January 11, 2026", "April 10, 2026", "January 08, 2026", "January 19, 2026",
        "January 24, 2026", "May 05, 2026", "January 30, 2026", "January 14, 2026", "January 02, 2026",
        "January 29, 2026", "January 17, 2026", "January 21, 2026"
    ]
}

# Load dataframe
df = pd.DataFrame(raw_data)

# =====================================================================
# 2. DATA PROCESSING & CONDITIONAL FILTERING
# =====================================================================
# Extract numeric string values from file sizes
df["Size_Numeric"] = df["Size"].astype(str).str.replace("M", "", regex=False)
df["Size_Numeric"] = pd.to_numeric(df["Size_Numeric"], errors="coerce")

# Convert datatypes to clean numeric categories
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")
df["Installs"] = pd.to_numeric(df["Installs"], errors="coerce")

# Date conversions for time-series extraction
df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce")
df["Month"] = df["Last Updated"].dt.strftime("%B")

# Apply target filters: Rating >= 4.0, Size >= 10M (or device dependent), and updated in January
filtered_df = df[
    (df["Rating"] >= 4.0) & 
    ((df["Size_Numeric"] >= 10.0) | (df["Size"] == "Varies with device")) & 
    (df["Month"] == "January")
]

# Identify the Top 10 app categories aggregated based on maximum user Installs 
top_categories = filtered_df.groupby("Category")["Installs"].sum().reset_index()
top_10_categories_list = top_categories.sort_values(by="Installs", ascending=False).head(10)["Category"].tolist()

# Filter initial processed dataset down to matches in the Top 10 Category list
final_metric_df = filtered_df[filtered_df["Category"].isin(top_10_categories_list)]

# Aggregate matching category rows down to compute Mean Rating and Summed Review Volume
grouped_df = final_metric_df.groupby("Category").agg({
    "Rating": "mean",
    "Reviews": "sum"
}).reset_index()

# =====================================================================
# 3. DYNAMIC TIME-RESTRICTION WINDOW ENFORCEMENT (3:00 PM - 5:00 PM IST)
# =====================================================================
# Set active environment tracking timezone specifically to Indian Standard Time (IST)
ist_timezone = pytz.timezone("Asia/Kolkata")
current_time_ist = datetime.datetime.now(ist_timezone)
current_hour = current_time_ist.hour  

# 3:00 PM corresponds to hour 15, and 5:00 PM corresponds to hour 17 (exclusive)
# NOTE FOR TESTING: If testing outside 3-5 PM, lower the threshold bound values (e.g., replace 15 with 10) 
if 15 <= current_hour < 17:
    
    # Generate Multi-Axis interactive bar charts 
    fig = go.Figure()

    # Average Rating Data Mapping (Primary Left Vertical Axis)
    fig.add_trace(go.Bar(
        x=grouped_df["Category"],
        y=grouped_df["Rating"],
        name="Average Rating",
        yaxis="y1",
        marker_color="#1f77b4"
    ))

    # Total Reviews Data Mapping (Secondary Right Vertical Axis)
    fig.add_trace(go.Bar(
        x=grouped_df["Category"],
        y=grouped_df["Reviews"],
        name="Total Reviews",
        yaxis="y2",
        marker_color="#ff7f0e"
    ))

    # Apply configuration schemas defining twin axis system layout mechanics
    fig.update_layout(
        title="Task 1 Dashboard: Top App Categories (Avg Rating vs Total Reviews)",
        barmode="group",
        xaxis=dict(title="App Categories"),
        yaxis=dict(title="Average Rating (Scale 0-5)", side="left"),
        yaxis2=dict(title="Cumulative User Reviews Volume", side="right", overlaying="y", showgrid=False),
        legend=dict(x=1.1, y=1),
    )

    # Render interactive figure inline 
    fig.show()

else:
    # Safely clear visualization element outputs and render silent administrative string log outside access times
    print(f"🔒 Dashboard Status: Hidden. This visual is restricted. Current IST Hour is {current_hour}. It displays only between 3 PM and 5 PM IST.")

🔒 Dashboard Status: Hidden. This visual is restricted. Current IST Hour is 10. It displays only between 3 PM and 5 PM IST.
